# AURA — XLM-RoBERTa + LoRA/PEFT Backend

This notebook upgrades the AURA backend to use a **trained multilingual XLM-RoBERTa model with LoRA/PEFT** for six-label toxicity classification.

Pipeline:

**Streamlit Frontend → FastAPI Backend → XLM-RoBERTa + LoRA/PEFT → Multi-label Toxicity Classification → Context/Behaviour/Harassment/Escalation Risk → Explainable AI → Rule-Based Decision Engine → SQLite**

Labels:
- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

Training requires a real labelled CSV. The expected format is a `text` column plus the six binary label columns.

In [ ]:
%pip install -q "numpy==1.26.4" "pandas==2.2.2" "numexpr==2.10.2" "bottleneck==1.4.2" "h5py==3.11.0" "scipy==1.13.1" "scikit-learn==1.5.2"

In [1]:
# Cell 2 — Imports and configuration

import os
import re
import json
import sqlite3

import secrets
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional, Any

import numpy as np
import pandas as pd
import torch


from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

from peft import LoraConfig, TaskType, get_peft_model, PeftModel

from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

set_seed(42)

BASE_DIR = Path.cwd()
MODEL_NAME = "FacebookAI/xlm-roberta-base"

DATASET_PATH = BASE_DIR / "aura_multilabel_dataset.csv"
MODEL_DIR = BASE_DIR / "aura_xlmr_lora_model"
ADAPTER_DIR = MODEL_DIR / "adapter"
TOKENIZER_DIR = MODEL_DIR / "tokenizer"
METRICS_PATH = MODEL_DIR / "training_metrics.json"
LABELS_PATH = MODEL_DIR / "labels.json"
DB_PATH = BASE_DIR / "aura.db"

MAX_LENGTH = 160
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: XLM-R training on CPU can be very slow.")
print("Dataset:", DATASET_PATH)
print("Model:", MODEL_NAME)

PyTorch: 2.13.0+cpu
CUDA available: False
Dataset: C:\Users\Win\Downloads\aura_multilabel_dataset.csv
Model: FacebookAI/xlm-roberta-base


## Required dataset format

```text
text,toxic,severe_toxic,obscene,threat,insult,identity_hate
"Great work!",0,0,0,0,0,0
"You are stupid",1,0,0,0,1,0
"I will hurt you",1,0,0,1,0,0
```

A message may have multiple `1` values. That is why this is a **multi-label classification** problem.

In [2]:
# Cell 3 — Load and validate dataset

def load_multilabel_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path}\n"
            "Place the labelled CSV there or change DATASET_PATH."
        )

    df = pd.read_csv(path)
    required = ["text"] + LABELS
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}. Required: {required}")

    df = df[required].copy()
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    df = df[df["text"] != ""].drop_duplicates("text").reset_index(drop=True)

    for label in LABELS:
        df[label] = pd.to_numeric(df[label], errors="coerce").fillna(0).astype(int)
        if not df[label].isin([0, 1]).all():
            raise ValueError(f"{label} must contain only 0 and 1.")

    return df

if DATASET_PATH.exists():
    df = load_multilabel_dataset(DATASET_PATH)
    print("Rows:", len(df))
    display(df.head())
else:
    df = None
    print("Add the labelled CSV before training.")

Add the labelled CSV before training.


In [3]:
# Cell 4 — Inspect class distribution

if df is not None:
    for label in LABELS:
        print(f"{label:15s}: {int(df[label].sum())} positive samples")
    print("Shape:", df.shape)
    print("Average labels/message:", round(df[LABELS].sum(axis=1).mean(), 3))

In [4]:
# Cell 5 — Train/validation/test split

if df is not None:
    train_df, temp_df = train_test_split(
        df, test_size=0.20, random_state=42, shuffle=True
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, shuffle=True
    )

    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))

In [5]:
# Cell 6 — Hugging Face datasets

if df is not None:
    dataset = DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False),
        "validation": Dataset.from_pandas(val_df.reset_index(drop=True), preserve_index=False),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True), preserve_index=False),
    })
    print(dataset)

In [6]:
# Cell 7 — XLM-R tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )
    

if df is not None:
    tokenized = dataset.map(tokenize_batch, batched=True, desc="Tokenizing")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# Cell 8 — Prepare six-label target vector

def add_labels(example):
    example["labels"] = [float(example[label]) for label in LABELS]
    return example

if df is not None:
    tokenized = tokenized.map(add_labels)
    tokenized = tokenized.remove_columns(["text"] + LABELS)
    print(tokenized["train"].column_names)
    print("Example labels:", tokenized["train"][0]["labels"])

In [8]:
# Cell 9 — XLM-R + LoRA/PEFT model

id2label = {i: label.upper() for i, label in enumerate(LABELS)}
label2id = {label.upper(): i for i, label in enumerate(LABELS)}

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification",
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.10,
    bias="none",
    target_modules=["query", "value"],
    modules_to_save=["classifier"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 890,118 || all params: 278,938,380 || trainable%: 0.3191


In [12]:
# Cell 10 — Padding/data collator

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

In [13]:
# Cell 11 — Multi-label evaluation metrics

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = sigmoid(logits)
    predictions = (probabilities >= 0.5).astype(int)
    labels = labels.astype(int)

    return {
        "micro_f1": f1_score(labels, predictions, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, predictions, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, predictions, average="micro", zero_division=0),
    }

In [14]:
# Cell 12 — Training configuration

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    seed=42,
)

In [15]:
# Cell 13 — Trainer

if df is not None:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    print("Trainer ready.")
else:
    trainer = None
    print("Load a real dataset first.")

Load a real dataset first.


## Cell 14 — Actual training

This is the actual **XLM-RoBERTa + LoRA/PEFT training step**. Run it only after the real labelled dataset has been loaded.

Hugging Face `Trainer` provides the training/evaluation loop, while PEFT supplies the LoRA adapter. citeturn0search0turn0search13

In [19]:
import pandas as pd

DATASET_PATH = r"C:\Users\Win\Downloads\train.csv"

df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())
print(df.head())

Shape: (159571, 8)
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  


In [21]:
df = df.rename(columns={"comment_text": "text"})

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

df = df[["text"] + LABELS]

df.to_csv(
    r"C:\Users\Win\Downloads\aura_multilabel_dataset.csv",
    index=False
)

print("AURA dataset created successfully.")
print("Shape:", df.shape)

AURA dataset created successfully.
Shape: (159571, 7)


In [18]:
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nFiles in this folder:")
for f in os.listdir():
    print(f)

Current notebook folder:
C:\Users\Win\Downloads

Files in this folder:
.ipynb_checkpoints
03 Active Machine Learning for Heterogeneity Activity Recognition Through Smartwatch Sensors(20260806-083840).docx
0815 (1).mp4
0815.mp4
24a614c4-239a-4d85-a99d-194b10ecee06_medium (1).webp
24a614c4-239a-4d85-a99d-194b10ecee06_medium.webp
A Hybrid Machine Learning Framework for Electric Vehicle Battery Health Prediction Using Charging Patterns and Performance Analytics.docx
A Multimodal Learning Framework for Disaster Damage Assessment Using Satellite Images and Textual Reports.docx
A SMART MINING VEHICLE ACCESS CONTROL AND OVERLOAD MONITORING SYSTEM.docx
A SMART MINING VEHICLE ACCESS CONTROL AND OVERLOAD MONITORING SYSTEM.pdf
a-group-of-seven-cartoon-robots-are-standing-on-green-grass-smiling-and-looking-happy-free-vector.jpg
Airqualitypred.ipynb
air_quality_dataset.csv
AI_5_Page_Document_for_RAG_Testing.pdf
Aman_Gogo_awe_futuristic_space_202608011654.mp4
AN AI-POWERED UNIVERSITY ADMISSION GUIDAN

In [24]:
# ============================================================
# AURA - REBUILD DATASET + XLM-R + LoRA TRAINER
# Run this cell BEFORE the TRAIN cell
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from peft import LoraConfig, TaskType, get_peft_model


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_PATH = r"C:\Users\Win\Downloads\aura_multilabel_dataset.csv"

MODEL_NAME = "FacebookAI/xlm-roberta-base"

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

NUM_LABELS = len(LABELS)
MAX_LENGTH = 160

print("=" * 70)
print("AURA XLM-RoBERTa + LoRA TRAINING SETUP")
print("=" * 70)


# ============================================================
# 2. CHECK DATASET
# ============================================================

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )

print("\nDataset found:")
print(DATASET_PATH)


# ============================================================
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv(DATASET_PATH)

print("\nOriginal shape:", df.shape)

required_columns = ["text"] + LABELS

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

# Remove missing text
df = df.dropna(subset=["text"]).copy()

# Make text string
df["text"] = df["text"].astype(str)

# Ensure labels are numeric
for label in LABELS:
    df[label] = pd.to_numeric(
        df[label],
        errors="coerce"
    ).fillna(0).astype(int)

# Keep only required columns
df = df[required_columns]

print("Cleaned shape:", df.shape)

print("\nLabel distribution:")
print(df[LABELS].sum())


# ============================================================
# 4. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("\nDataset split:")
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))


# ============================================================
# 5. CONVERT TO HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


# ============================================================
# 6. CREATE MULTI-LABEL VECTOR
# ============================================================

def add_labels(example):

    example["labels"] = [
        float(example[label])
        for label in LABELS
    ]

    return example


train_dataset = train_dataset.map(add_labels)
val_dataset = val_dataset.map(add_labels)
test_dataset = test_dataset.map(add_labels)


# ============================================================
# 7. TOKENIZER
# ============================================================

print("\nLoading XLM-RoBERTa tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 8. TOKENIZATION
# ============================================================

def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


# ============================================================
# 9. REMOVE UNNECESSARY COLUMNS
# ============================================================

columns_to_remove = [
    "text",
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

train_dataset = train_dataset.remove_columns(
    [c for c in columns_to_remove if c in train_dataset.column_names]
)

val_dataset = val_dataset.remove_columns(
    [c for c in columns_to_remove if c in val_dataset.column_names]
)

test_dataset = test_dataset.remove_columns(
    [c for c in columns_to_remove if c in test_dataset.column_names]
)


# ============================================================
# 10. LOAD XLM-ROBERTA
# ============================================================

print("\nLoading XLM-RoBERTa model...")

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)


# ============================================================
# 11. CONFIGURE LoRA
# ============================================================

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,

    r=8,

    lora_alpha=16,

    lora_dropout=0.10,

    bias="none",

    target_modules=[
        "query",
        "value"
    ],

    modules_to_save=[
        "classifier"
    ]
)


# ============================================================
# 12. APPLY LoRA
# ============================================================

model = get_peft_model(
    base_model,
    lora_config
)

print("\nLoRA model created.")

model.print_trainable_parameters()


# ============================================================
# 13. DATA COLLATOR
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ============================================================
# 14. METRICS
# ============================================================

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probabilities = 1 / (
        1 + np.exp(-logits)
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    labels = labels.astype(int)

    return {
        "micro_f1": f1_score(
            labels,
            predictions,
            average="micro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "micro_precision": precision_score(
            labels,
            predictions,
            average="micro",
            zero_division=0
        ),

        "micro_recall": recall_score(
            labels,
            predictions,
            average="micro",
            zero_division=0
        )
    }


# ============================================================
# 15. TRAINING ARGUMENTS
# ============================================================

OUTPUT_DIR = r"C:\Users\Win\Downloads\aura_xlmr_lora_model"

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    learning_rate=2e-4,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="micro_f1",

    greater_is_better=True,

    save_total_limit=2,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=0
)


# ============================================================
# 16. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


# ============================================================
# 17. FINAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("TRAINER CREATED SUCCESSFULLY")
print("=" * 70)

print("Trainer:", trainer)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

print("\nYou can now run the TRAIN cell.")

AURA XLM-RoBERTa + LoRA TRAINING SETUP

Dataset found:
C:\Users\Win\Downloads\aura_multilabel_dataset.csv

Original shape: (159571, 7)
Cleaned shape: (159571, 7)

Label distribution:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Dataset split:
Train      : 127656
Validation : 15957
Test       : 15958


Map:   0%|          | 0/127656 [00:00<?, ? examples/s]

Map:   0%|          | 0/15957 [00:00<?, ? examples/s]

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]


Loading XLM-RoBERTa tokenizer...


...

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]


Loading XLM-RoBERTa model...


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



LoRA model created.
trainable params: 890,118 || all params: 278,938,380 || trainable%: 0.3191

TRAINER CREATED SUCCESSFULLY
Trainer: <transformers.trainer.Trainer object at 0x0000027F44F79950>
Train samples: 127656
Validation samples: 15957
Test samples: 15958

You can now run the TRAIN cell.


C:\Users\Win\AppData\Local\Temp\ipykernel_6200\4258264822.py:400: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [26]:
# ============================================================
# CELL 14 — FAST TRAINING
# ============================================================

# Use only a smaller subset for faster CPU training
FAST_TRAIN_SIZE = 8000
FAST_VAL_SIZE = 1000

# Create smaller datasets
train_df_fast = train_df.sample(
    n=min(FAST_TRAIN_SIZE, len(train_df)),
    random_state=42
)

val_df_fast = val_df.sample(
    n=min(FAST_VAL_SIZE, len(val_df)),
    random_state=42
)

print("Fast dataset:")
print("Train:", len(train_df_fast))
print("Validation:", len(val_df_fast))


# ============================================================
# CREATE HUGGING FACE DATASETS
# ============================================================

from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df_fast,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df_fast,
    preserve_index=False
)


# ============================================================
# ADD MULTI-LABEL TARGETS
# ============================================================

def add_labels(example):
    example["labels"] = [
        float(example[label])
        for label in LABELS
    ]
    return example


train_dataset = train_dataset.map(add_labels)
val_dataset = val_dataset.map(add_labels)


# ============================================================
# TOKENIZATION
# ============================================================

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)


# ============================================================
# REMOVE ORIGINAL LABEL COLUMNS
# ============================================================

remove_columns = [
    "text",
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

for column in remove_columns:

    if column in train_dataset.column_names:
        train_dataset = train_dataset.remove_columns(column)

    if column in val_dataset.column_names:
        val_dataset = val_dataset.remove_columns(column)


# ============================================================
# FAST TRAINING SETTINGS
# ============================================================

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    # ONLY 1 EPOCH
    num_train_epochs=1,

    # CPU-friendly
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=1,

    report_to="none",

    # CPU system
    fp16=False,

    dataloader_num_workers=0,

    dataloader_pin_memory=False
)


# ============================================================
# CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


# ============================================================
# START TRAINING
# ============================================================

print("\nStarting FAST XLM-R + LoRA training...")

train_result = trainer.train()

print("\nTraining completed successfully!")
print(train_result)

Fast dataset:
Train: 8000
Validation: 1000


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

C:\Users\Win\AppData\Local\Temp\ipykernel_6200\819382116.py:149: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Starting FAST XLM-R + LoRA training...


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,0.095000,0.062882,0.661290,0.351654,0.750000,0.591346


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a78abc00-4c7e-44c1-a83f-e7a8c938319a)')' thrown while requesting HEAD https://huggingface.co/FacebookAI/xlm-roberta-base/resolve/main/config.json
Retrying in 1s [Retry 1/5].



Training completed successfully!
TrainOutput(global_step=1000, training_loss=0.08150200176239014, metrics={'train_runtime': 2940.8093, 'train_samples_per_second': 2.72, 'train_steps_per_second': 0.34, 'total_flos': 511450916872896.0, 'train_loss': 0.08150200176239014, 'epoch': 1.0})


In [28]:
# ============================================================
# CELL 15 — FAST TEST EVALUATION
# ============================================================

from datasets import Dataset

# ------------------------------------------------------------
# Create a small test set for fast evaluation
# ------------------------------------------------------------

FAST_TEST_SIZE = 1000

test_df_fast = test_df.sample(
    n=min(FAST_TEST_SIZE, len(test_df)),
    random_state=42
)

print("Test samples:", len(test_df_fast))


# ------------------------------------------------------------
# Convert to Hugging Face Dataset
# ------------------------------------------------------------

test_dataset = Dataset.from_pandas(
    test_df_fast,
    preserve_index=False
)


# ------------------------------------------------------------
# Add multi-label targets
# ------------------------------------------------------------

def add_test_labels(example):

    example["labels"] = [
        float(example[label])
        for label in LABELS
    ]

    return example


test_dataset = test_dataset.map(
    add_test_labels
)


# ------------------------------------------------------------
# Tokenize test data
# ------------------------------------------------------------

def tokenize_test(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )


test_dataset = test_dataset.map(
    tokenize_test,
    batched=True
)


# ------------------------------------------------------------
# Remove original columns
# ------------------------------------------------------------

remove_columns = [
    "text",
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

for column in remove_columns:

    if column in test_dataset.column_names:
        test_dataset = test_dataset.remove_columns(column)


# ------------------------------------------------------------
# Evaluate trained model
# ------------------------------------------------------------

test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("AURA MODEL TEST RESULTS")
print("=" * 60)

for key, value in test_metrics.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Test samples: 1000


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


AURA MODEL TEST RESULTS
test_loss: 0.0674
test_micro_f1: 0.6828
test_macro_f1: 0.3463
test_micro_precision: 0.6769
test_micro_recall: 0.6889
test_runtime: 103.9751
test_samples_per_second: 9.6180
test_steps_per_second: 1.2020
epoch: 1.0000


In [29]:
# Cell 16 — Save LoRA adapter and tokenizer

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)

print("Adapter:", ADAPTER_DIR)
print("Tokenizer:", TOKENIZER_DIR)

Adapter: C:\Users\Win\Downloads\aura_xlmr_lora_model\adapter
Tokenizer: C:\Users\Win\Downloads\aura_xlmr_lora_model\tokenizer


In [30]:
# Cell 17 — Load trained XLM-R + LoRA model for inference

def load_trained_model():
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(
            f"No LoRA adapter found at {ADAPTER_DIR}. Train the model first."
        )

    inference_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)

    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label={i: label.upper() for i, label in enumerate(LABELS)},
        label2id={label.upper(): i for i, label in enumerate(LABELS)},
        problem_type="multi_label_classification",
    )

    inference_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    inference_model.eval()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inference_model.to(device)

    return inference_model, inference_tokenizer, device

inference_model, inference_tokenizer, inference_device = load_trained_model()
print("Trained model loaded on:", inference_device)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trained model loaded on: cpu


In [31]:
# Cell 18 — Model prediction

def model_predict(text, model_obj, tokenizer_obj, device_obj, threshold=0.50):
    inputs = tokenizer_obj(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(device_obj) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model_obj(**inputs).logits

    probabilities = torch.sigmoid(logits)[0].cpu().numpy()

    scores = {
        label: float(probabilities[i] * 100.0)
        for i, label in enumerate(LABELS)
    }

    detected = {
        label: score
        for label, score in scores.items()
        if score >= threshold * 100.0
    }

    return scores, detected

## Context and safety layers

The trained model is now the primary toxicity classifier. The following deterministic layers implement the additional AURA architecture features: language handling, context, behaviour, harassment, escalation, risk aggregation, explainability, and moderation policy.

In [32]:
# Cell 19 — Text normalization and language detection

def norm(text: str) -> str:
    text = str(text or "").strip().lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

ROMAN_MARKERS = (
    "muttal", "moodan", "kevalam", "asingam", "thodarnthu",
    "meendum meendum", "ennai", "unnai", "harass",
    "vedhava", "pichivadu", "nuvvu", "nannu", "ninnu",
    "malli malli", "tarachuga", "moorkha", "dadda",
    "neenu", "nannannu", "ninnannu", "matte matte",
    "mandan", "viddhi", "ninne", "veendum veendum",
    "bewakoof", "moorkh", "gadha", "ghatiya", "tum",
    "mujhe", "tumhe", "bar bar", "baar baar", "lagatar",
    "boka", "murkho", "tumi", "amake", "tomake", "barbar",
)

def detect_language(text: str) -> dict:
    t = norm(text)

    scripts = {
        "hindi": r"[\u0900-\u097F]",
        "bengali": r"[\u0980-\u09FF]",
        "tamil": r"[\u0B80-\u0BFF]",
        "telugu": r"[\u0C00-\u0C7F]",
        "kannada": r"[\u0C80-\u0CFF]",
        "malayalam": r"[\u0D00-\u0D7F]",
    }

    detected = [name for name, pattern in scripts.items() if re.search(pattern, t)]
    romanized = any(marker in t for marker in ROMAN_MARKERS)

    if detected:
        primary = detected[0]
        confidence = 0.95
    elif romanized:
        primary = "romanized/code-mixed"
        confidence = 0.75
    else:
        primary = "english/latin"
        confidence = 0.85

    return {
        "primary": primary,
        "detected_scripts": detected,
        "code_mixed": bool(romanized and detected) or romanized,
        "confidence": confidence,
    }

In [33]:
# Cell 20 — Context, harassment and escalation rules

HARASSMENT_CUES = [
    "targeting me", "keep targeting", "stop targeting",
    "harassing me", "harass me", "keep bothering",
    "stop bothering", "repeatedly message", "leave me alone",
    "nobody wants you around",
]

ESCALATION_CUES = [
    "again", "every discussion", "repeatedly",
    "keep doing this", "keep contacting me",
    "next time", "you will regret", "you'll regret",
]

THREAT_CUES = [
    "i will hurt you", "i'll hurt you", "i will kill you",
    "i'll kill you", "kill you", "find you and hurt",
    "attack you", "beat you",
]

AMBIGUOUS_CUES = [
    "maybe something will happen", "you are going to regret",
    "you'll regret", "you will regret", "maybe i should",
    "i don't know what i'll do", "i am not sure",
    "you better watch",
]

def count_cues(text, cues):
    t = norm(text)
    return sum(1 for cue in cues if cue in t)

def harassment_risk(text, context):
    return min(100.0, 25.0 * count_cues(text, HARASSMENT_CUES) + 10.0 * len(context))

def escalation_risk(text):
    return min(100.0, 20.0 * count_cues(text, ESCALATION_CUES))

def direct_threat_risk(text):
    return min(100.0, 35.0 * count_cues(text, THREAT_CUES))

def ambiguity_risk(text):
    return min(100.0, 30.0 * count_cues(text, AMBIGUOUS_CUES))

In [34]:
# Cell 21 — SQLite database

def db():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def now():
    return datetime.now(timezone.utc).isoformat()

def init_db():
    conn = db()
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            email TEXT UNIQUE,
            password_hash TEXT NOT NULL,
            salt TEXT NOT NULL,
            role TEXT NOT NULL DEFAULT 'user',
            blocked INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS moderation_results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            conversation_id TEXT NOT NULL,
            message TEXT NOT NULL,
            result_json TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS human_reviews (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            moderation_id INTEGER NOT NULL,
            moderator_id INTEGER,
            decision TEXT NOT NULL,
            note TEXT,
            created_at TEXT NOT NULL
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS audit_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            action TEXT NOT NULL,
            details TEXT,
            created_at TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()

init_db()
print("Database ready:", DB_PATH)

Database ready: C:\Users\Win\Downloads\aura.db


In [35]:
# Cell 22 — Authentication/session security

SESSIONS: dict[str, dict[str, Any]] = {}

def hash_password(password, salt=None):
    salt = salt or secrets.token_hex(16)
    hashed = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        salt.encode("utf-8"),
        150000,
    ).hex()
    return hashed, salt

def verify_password(password, stored_hash, salt):
    hashed, _ = hash_password(password, salt)
    return secrets.compare_digest(hashed, stored_hash)

def create_token(username, role, user_id):
    token = secrets.token_urlsafe(32)
    SESSIONS[token] = {
        "username": username,
        "role": role,
        "user_id": user_id,
        "created_at": now(),
    }
    return token

def current_user(authorization: Optional[str]):
    if not authorization or not authorization.lower().startswith("bearer "):
        raise HTTPException(status_code=401, detail="Authentication required")

    token = authorization.split(" ", 1)[1].strip()
    session = SESSIONS.get(token)

    if not session:
        raise HTTPException(status_code=401, detail="Invalid or expired session")

    return session

def admin_user(authorization: Optional[str]):
    user = current_user(authorization)
    if user["role"] != "admin":
        raise HTTPException(status_code=403, detail="Admin access required")
    return user

In [36]:
# Cell 23 — Risk engine and final moderation decision

def get_user_history_stats(user_id):
    if not user_id:
        return {"previous_violations": 0, "previous_warnings": 0}

    conn = db()
    rows = conn.execute(
        "SELECT result_json FROM moderation_results WHERE user_id = ?",
        (user_id,),
    ).fetchall()
    conn.close()

    violations = 0
    warnings = 0

    for row in rows:
        try:
            result = json.loads(row["result_json"])
            action = result.get("action", "")
            if action in {"QUARANTINE", "RESTRICT", "TEMPORARY_BLOCK"}:
                violations += 1
            if action == "WARN":
                warnings += 1
        except Exception:
            pass

    return {
        "previous_violations": violations,
        "previous_warnings": warnings,
    }

def build_risk_result(text, model_scores, context, history):
    max_model = max(model_scores.values()) if model_scores else 0.0

    harassment = harassment_risk(text, context)
    escalation = escalation_risk(text)
    direct_threat = direct_threat_risk(text)
    ambiguity = ambiguity_risk(text)

    previous_violations = history.get("previous_violations", 0)
    previous_warnings = history.get("previous_warnings", 0)

    behavioural = min(
        100.0,
        previous_violations * 20.0 + previous_warnings * 5.0,
    )

    context_risk = min(100.0, len(context) * 8.0)

    overall = (
        0.60 * max_model
        + 0.10 * context_risk
        + 0.10 * behavioural
        + 0.10 * harassment
        + 0.10 * escalation
    )

    overall = max(overall, direct_threat)

    if ambiguity >= 30 and max_model < 60:
        action = "HUMAN_REVIEW"
    elif direct_threat >= 70 or model_scores.get("threat", 0) >= 85:
        action = "TEMPORARY_BLOCK"
    elif harassment >= 60 or previous_violations >= 2:
        action = "RESTRICT"
    elif (
        model_scores.get("severe_toxic", 0) >= 80
        or model_scores.get("obscene", 0) >= 75
        or model_scores.get("identity_hate", 0) >= 80
        or overall >= 65
    ):
        action = "QUARANTINE"
    elif (
        model_scores.get("toxic", 0) >= 35
        or model_scores.get("insult", 0) >= 35
        or overall >= 30
    ):
        action = "WARN"
    else:
        action = "ALLOW"

    return {
        "model_scores": {k: round(float(v), 2) for k, v in model_scores.items()},
        "overall_risk": round(float(overall), 2),
        "context_risk": round(float(context_risk), 2),
        "behavioural_risk": round(float(behavioural), 2),
        "harassment_risk": round(float(harassment), 2),
        "escalation_risk": round(float(escalation), 2),
        "direct_threat_risk": round(float(direct_threat), 2),
        "ambiguity_risk": round(float(ambiguity), 2),
        "previous_violations": previous_violations,
        "previous_warnings": previous_warnings,
        "action": action,
    }

In [37]:
# Cell 24 — Explainable AI

def build_explanation(text, model_scores, risk_result):
    evidence = []

    for label, score in sorted(
        model_scores.items(), key=lambda item: item[1], reverse=True
    ):
        if score >= 50:
            evidence.append(
                f"{label.replace('_', ' ').title()} model score: {score:.1f}%"
            )

    if risk_result["harassment_risk"] > 0:
        evidence.append(
            f"Harassment risk: {risk_result['harassment_risk']:.1f}%"
        )

    if risk_result["escalation_risk"] > 0:
        evidence.append(
            f"Escalation risk: {risk_result['escalation_risk']:.1f}%"
        )

    if risk_result["previous_violations"] > 0:
        evidence.append(
            f"Previous violations: {risk_result['previous_violations']}"
        )

    if not evidence:
        evidence.append("No high-risk evidence exceeded the explanation threshold.")

    return {
        "important_evidence": evidence,
        "summary": f"AURA classified the message as {risk_result['action']}.",
        "steps": [
            "XLM-RoBERTa + LoRA generated six safety probabilities.",
            "Context, behaviour, harassment and escalation rules were evaluated.",
            "The risk engine combined the factors.",
            f"The decision engine selected {risk_result['action']}.",
        ],
    }

In [38]:
# Cell 25 — FastAPI application and CORS

app = FastAPI(
    title="AURA Intelligent Content Safety API",
    version="2.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://127.0.0.1:8501",
        "http://localhost:8501",
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [39]:
# Cell 26 — FastAPI request models

class LoginIn(BaseModel):
    username: str
    password: str
    role: Optional[str] = None

class RegisterIn(BaseModel):
    username: str
    password: str
    email: Optional[str] = None

class MessageIn(BaseModel):
    text: Optional[str] = None
    message: Optional[str] = None
    context: str = ""
    context_messages: list[str] = Field(default_factory=list)
    user_id: str = "anonymous"
    conversation_id: str = "default"

    def get_text(self):
        return (self.text if self.text is not None else self.message or "").strip()

    def get_context(self):
        return (
            [str(x).strip() for x in self.context_messages if str(x).strip()]
            or [x.strip() for x in self.context.splitlines() if x.strip()]
        )

class HumanDecisionIn(BaseModel):
    decision: str
    note: str = ""

class BlockIn(BaseModel):
    username: str

In [40]:
# Cell 27 — Root, health, registration and login

@app.get("/")
def root():
    return {
        "name": "AURA",
        "status": "online",
        "model": MODEL_NAME,
        "adapter": "LoRA/PEFT",
    }

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_available": ADAPTER_DIR.exists(),
        "model": MODEL_NAME,
        "adapter": "LoRA/PEFT",
    }

@app.post("/auth/register")
def register(req: RegisterIn):
    username = req.username.strip()

    if len(username) < 3:
        raise HTTPException(status_code=400, detail="Username must be at least 3 characters.")
    if len(req.password) < 6:
        raise HTTPException(status_code=400, detail="Password must be at least 6 characters.")

    password_hash, salt = hash_password(req.password)

    conn = db()
    try:
        cur = conn.execute(
            """
            INSERT INTO users
            (username, email, password_hash, salt, role, created_at)
            VALUES (?, ?, ?, ?, 'user', ?)
            """,
            (username, req.email, password_hash, salt, now()),
        )
        conn.commit()
        user_id = cur.lastrowid
    except sqlite3.IntegrityError:
        conn.close()
        raise HTTPException(status_code=409, detail="Username or email already exists.")
    conn.close()

    return {"success": True, "message": "Account created.", "user_id": user_id}

@app.post("/auth/login")
def login(req: LoginIn):
    conn = db()
    row = conn.execute(
        "SELECT * FROM users WHERE username = ?",
        (req.username.strip(),),
    ).fetchone()
    conn.close()

    if not row or not verify_password(req.password, row["password_hash"], row["salt"]):
        raise HTTPException(status_code=401, detail="Invalid username or password.")

    if row["blocked"]:
        raise HTTPException(status_code=403, detail="User is blocked.")

    if req.role and req.role != row["role"]:
        raise HTTPException(status_code=403, detail="Role mismatch.")

    token = create_token(row["username"], row["role"], row["id"])

    return {
        "success": True,
        "token": token,
        "username": row["username"],
        "role": row["role"],
        "user_id": row["id"],
    }

In [41]:
# Cell 28 — Main moderation endpoint

@app.post("/moderate")
def moderate(
    req: MessageIn,
    authorization: Optional[str] = Header(default=None),
):
    user = current_user(authorization)

    if inference_model is None:
        raise HTTPException(
            status_code=503,
            detail="Trained XLM-R + LoRA model is not loaded.",
        )

    text = req.get_text()
    if not text:
        raise HTTPException(status_code=400, detail="Message text is required.")

    context = req.get_context()

    model_scores, detected = model_predict(
        text,
        inference_model,
        inference_tokenizer,
        inference_device,
    )

    history = get_user_history_stats(user["user_id"])

    risk = build_risk_result(
        text=text,
        model_scores=model_scores,
        context=context,
        history=history,
    )

    language = detect_language(text)
    explanation = build_explanation(text, model_scores, risk)

    result = {
        "message": text,
        "language": language,
        "detected_labels": detected,
        "model_scores": model_scores,
        "risk": risk,
        "action": risk["action"],
        "status": (
            "POSTED"
            if risk["action"] == "ALLOW"
            else (
                "PENDING HUMAN REVIEW"
                if risk["action"] == "HUMAN_REVIEW"
                else "NOT POSTED"
            )
        ),
        "visible_to_public": risk["action"] == "ALLOW",
        "needs_human": risk["action"] == "HUMAN_REVIEW",
        "explainable_ai": explanation,
        "created_at": now(),
    }

    conn = db()

    cur = conn.execute(
        """
        INSERT INTO moderation_results
        (user_id, conversation_id, message, result_json, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            user["user_id"],
            req.conversation_id,
            text,
            json.dumps(result, ensure_ascii=False),
            now(),
        ),
    )

    conn.execute(
        """
        INSERT INTO audit_logs
        (user_id, action, details, created_at)
        VALUES (?, ?, ?, ?)
        """,
        (
            user["user_id"],
            "MODERATION",
            json.dumps({"action": risk["action"]}, ensure_ascii=False),
            now(),
        ),
    )

    conn.commit()
    moderation_id = cur.lastrowid
    conn.close()

    result["moderation_id"] = moderation_id
    return result

In [42]:
# Cell 29 — User history

@app.get("/history")
def history(authorization: Optional[str] = Header(default=None)):
    user = current_user(authorization)

    conn = db()
    rows = conn.execute(
        """
        SELECT id, conversation_id, message, result_json, created_at
        FROM moderation_results
        WHERE user_id = ?
        ORDER BY id DESC
        """,
        (user["user_id"],),
    ).fetchall()
    conn.close()

    return {
        "items": [
            {
                "id": row["id"],
                "conversation_id": row["conversation_id"],
                "message": row["message"],
                "action": json.loads(row["result_json"]).get("action"),
                "status": json.loads(row["result_json"]).get("status"),
                "model_scores": json.loads(row["result_json"]).get("model_scores"),
                "created_at": row["created_at"],
            }
            for row in rows
        ]
    }

In [43]:
# Cell 30 — Admin overview and comments

@app.get("/admin/overview")
def admin_overview(authorization: Optional[str] = Header(default=None)):
    admin_user(authorization)

    conn = db()
    total_users = conn.execute("SELECT COUNT(*) AS n FROM users").fetchone()["n"]
    total_moderations = conn.execute(
        "SELECT COUNT(*) AS n FROM moderation_results"
    ).fetchone()["n"]

    action_counts = {}
    rows = conn.execute("SELECT result_json FROM moderation_results").fetchall()
    conn.close()

    for row in rows:
        try:
            action = json.loads(row["result_json"]).get("action", "UNKNOWN")
            action_counts[action] = action_counts.get(action, 0) + 1
        except Exception:
            pass

    return {
        "total_users": total_users,
        "total_moderations": total_moderations,
        "actions": action_counts,
    }

@app.get("/admin/comments")
def admin_comments(authorization: Optional[str] = Header(default=None)):
    admin_user(authorization)

    conn = db()
    rows = conn.execute(
        """
        SELECT id, conversation_id, message, result_json, created_at
        FROM moderation_results
        ORDER BY id DESC
        """
    ).fetchall()
    conn.close()

    return {
        "items": [
            {
                "id": row["id"],
                "conversation_id": row["conversation_id"],
                "message": row["message"],
                "result": json.loads(row["result_json"]),
                "created_at": row["created_at"],
            }
            for row in rows
        ]
    }

In [44]:
# Cell 31 — Human review

@app.get("/admin/human-review")
def human_review_queue(authorization: Optional[str] = Header(default=None)):
    admin_user(authorization)

    conn = db()
    rows = conn.execute(
        """
        SELECT id, message, result_json, created_at
        FROM moderation_results
        WHERE json_extract(result_json, '$.action') = 'HUMAN_REVIEW'
        ORDER BY id DESC
        """
    ).fetchall()
    conn.close()

    return {
        "items": [
            {
                "moderation_id": row["id"],
                "message": row["message"],
                "result": json.loads(row["result_json"]),
                "created_at": row["created_at"],
            }
            for row in rows
        ]
    }

@app.post("/admin/human-review/{moderation_id}")
def human_review_decision(
    moderation_id: int,
    req: HumanDecisionIn,
    authorization: Optional[str] = Header(default=None),
):
    moderator = admin_user(authorization)

    allowed = {
        "APPROVE", "REJECT", "QUARANTINE",
        "RESTRICT", "BLOCK", "DISMISS"
    }

    decision = req.decision.strip().upper()
    if decision not in allowed:
        raise HTTPException(
            status_code=400,
            detail=f"Decision must be one of {sorted(allowed)}"
        )

    action_map = {
        "APPROVE": "ALLOW",
        "REJECT": "QUARANTINE",
        "QUARANTINE": "QUARANTINE",
        "RESTRICT": "RESTRICT",
        "BLOCK": "TEMPORARY_BLOCK",
        "DISMISS": "ALLOW",
    }

    conn = db()
    row = conn.execute(
        "SELECT result_json FROM moderation_results WHERE id = ?",
        (moderation_id,),
    ).fetchone()

    if not row:
        conn.close()
        raise HTTPException(status_code=404, detail="Moderation result not found.")

    result = json.loads(row["result_json"])
    result["action"] = action_map[decision]
    result["human_review"] = {
        "decision": decision,
        "note": req.note,
        "moderator_id": moderator["user_id"],
        "reviewed_at": now(),
    }

    conn.execute(
        "UPDATE moderation_results SET result_json = ? WHERE id = ?",
        (json.dumps(result, ensure_ascii=False), moderation_id),
    )

    conn.execute(
        """
        INSERT INTO human_reviews
        (moderation_id, moderator_id, decision, note, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        (moderation_id, moderator["user_id"], decision, req.note, now()),
    )

    conn.commit()
    conn.close()

    return {
        "success": True,
        "moderation_id": moderation_id,
        "decision": decision,
        "action": result["action"],
    }

In [45]:
# Cell 32 — Admin users and block user

@app.get("/admin/users")
def admin_users(authorization: Optional[str] = Header(default=None)):
    admin_user(authorization)

    conn = db()
    rows = conn.execute(
        """
        SELECT id, username, email, role, blocked, created_at
        FROM users ORDER BY id DESC
        """
    ).fetchall()
    conn.close()

    return {"items": [dict(row) for row in rows]}

@app.post("/admin/block-user")
def block_user(
    req: BlockIn,
    authorization: Optional[str] = Header(default=None),
):
    admin = admin_user(authorization)

    conn = db()
    cur = conn.execute(
        "UPDATE users SET blocked = 1 WHERE username = ?",
        (req.username.strip(),),
    )

    if cur.rowcount == 0:
        conn.close()
        raise HTTPException(status_code=404, detail="User not found.")

    conn.execute(
        """
        INSERT INTO audit_logs
        (user_id, action, details, created_at)
        VALUES (?, ?, ?, ?)
        """,
        (
            admin["user_id"],
            "BLOCK_USER",
            json.dumps({"username": req.username.strip()}),
            now(),
        ),
    )

    conn.commit()
    conn.close()

    return {"success": True, "message": f"User '{req.username.strip()}' blocked."}

In [46]:
# Cell 33 — Quick rule-layer smoke test

samples = [
    "Great work!",
    "You are stupid",
    "I will hurt you",
    "You keep targeting me in every discussion. Stop harassing me.",
]

for sample in samples:
    print("\nTEXT:", sample)
    print("LANGUAGE:", detect_language(sample))
    print("HARASSMENT:", harassment_risk(sample, []))
    print("ESCALATION:", escalation_risk(sample))
    print("THREAT:", direct_threat_risk(sample))


TEXT: Great work!
LANGUAGE: {'primary': 'english/latin', 'detected_scripts': [], 'code_mixed': False, 'confidence': 0.85}
HARASSMENT: 0.0
ESCALATION: 0.0
THREAT: 0.0

TEXT: You are stupid
LANGUAGE: {'primary': 'english/latin', 'detected_scripts': [], 'code_mixed': False, 'confidence': 0.85}
HARASSMENT: 0.0
ESCALATION: 0.0
THREAT: 0.0

TEXT: I will hurt you
LANGUAGE: {'primary': 'english/latin', 'detected_scripts': [], 'code_mixed': False, 'confidence': 0.85}
HARASSMENT: 0.0
ESCALATION: 0.0
THREAT: 35.0

TEXT: You keep targeting me in every discussion. Stop harassing me.
LANGUAGE: {'primary': 'romanized/code-mixed', 'detected_scripts': [], 'code_mixed': True, 'confidence': 0.75}
HARASSMENT: 75.0
ESCALATION: 20.0
THREAT: 0.0


In [48]:
# ============================================================
# AURA - MODEL ACCURACY / PERFORMANCE
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("=" * 60)
print("AURA XLM-RoBERTa + LoRA MODEL PERFORMANCE")
print("=" * 60)

# Make predictions on the test dataset
prediction_output = trainer.predict(test_dataset)

logits = prediction_output.predictions

# Convert logits to probabilities
probabilities = 1 / (1 + np.exp(-logits))

# Convert probabilities to 0/1 predictions
predictions = (probabilities >= 0.5).astype(int)

# True labels
true_labels = np.array(
    test_dataset["labels"],
    dtype=int
)

# ------------------------------------------------------------
# EXACT MATCH ACCURACY
# ------------------------------------------------------------

exact_accuracy = np.mean(
    np.all(predictions == true_labels, axis=1)
)

# ------------------------------------------------------------
# MICRO / MACRO F1
# ------------------------------------------------------------

micro_f1 = f1_score(
    true_labels,
    predictions,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    true_labels,
    predictions,
    average="macro",
    zero_division=0
)

micro_precision = precision_score(
    true_labels,
    predictions,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    true_labels,
    predictions,
    average="micro",
    zero_division=0
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print(f"Exact Match Accuracy : {exact_accuracy * 100:.2f}%")


print("=" * 60)

AURA XLM-RoBERTa + LoRA MODEL PERFORMANCE


Exact Match Accuracy : 90.90%


## Start the FastAPI backend

After training and loading the model, execute the next cell.

For normal project use, the equivalent terminal command is:

```bat
conda activate toxic_ai
python -m uvicorn main:app --reload
```

The Streamlit frontend can continue running separately on port 8501.

In [ ]:
# Cell 34 — Start Uvicorn inside Jupyter

import uvicorn

uvicorn.run(app, host="127.0.0.1", port=8000)

In [1]:
from __future__ import annotations
import hashlib, json, re, secrets, sqlite3
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

BASE_DIR = Path(__file__).resolve().parent
DB_PATH = BASE_DIR / 'aura.db'
app = FastAPI(title='AURA — Intelligent Content Safety', version='2.0.0')
app.add_middleware(CORSMiddleware, allow_origins=['http://127.0.0.1:8501','http://localhost:8501'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])
SESSIONS: dict[str, dict[str, Any]] = {}
LABELS=['toxic','severe_toxic','obscene','threat','insult','identity_hate']

# Multilingual safety lexicon.  English behaviour is preserved; these additions
# make the same six safety signals understandable in Indian scripts, Romanized
# Indian speech (Tanglish/Hinglish/etc.), and mixed-language comments.
LEX={
 'toxic':[
  # English
  'idiot','moron','dumb','stupid','shut up','useless','trash','garbage','loser','fool','jerk','clown','pathetic',
  # Tamil + Tanglish
  'பைத்தியம்','முட்டாள்','மூடன்','அறிவில்லாத','உபயோகமில்லை','வேஸ்ட்','கேவலமானவன்','கேவலமானவள்','கேவலமான ஆள்','அசிங்கமானவன்','அசிங்கமானவள்','கிறுக்கன்','முட்டாள்தனமாக',
  'paithiyam','muttal','muttal','moodan','arivillatha','use illa','useless','kevalamanavan','kevalamana aal','asingamanavan','asingamanaval','kirukkan',
  # Telugu + Tenglish
  'పిచ్చివాడు','పిచ్చిదానివి','మూర్ఖుడు','వెధవ','దరిద్రుడు','నీచుడు','పిచ్చి','వెధవా','వెధవుడు',
  'pichivadu','pichhodu','murkhudu','vedhava','daridrudu','nichudu','vedhavodini',
  # Kannada + Kanglish
  'ಹುಚ್ಚ','ಮೂರ್ಖ','ದಡ್ಡ','ಬೇವಕೂಫ್','ಕೆಟ್ಟವನು','ನೀಚ','ಹುಚ್ಚನ',
  'huchcha','moorkha','dadda','bevakuf','kettavanu','neech','huchchan',
  # Malayalam + Manglish
  'ഭ്രാന്തൻ','മണ്ടൻ','മണ്ടി','വിഡ്ഢി','നീചൻ','വൃത്തികെട്ടവൻ','പൊട്ടൻ',
  'bhranthan','mandan','mandi','viddhi','neechan','vrithikettavan','pottan',
  # Hindi + Hinglish
  'बेवकूफ','मूर्ख','गधा','पागल','निकम्मा','नालायक','घटिया','बदतमीज',
  'bewakoof','bewaqoof','moorkh','gadha','pagal','nikamma','nalayak','ghatiya','badtameez',
  # Bengali + Banglish
  'বোকা','মূর্খ','গাধা','পাগল','অযোগ্য','নিকম্মা','জঘন্য','অসভ্য',
  'boka','murkho','gadha','pagol','ojoggo','nikamma','joghonyo','oshobbo'
 ],
 'severe_toxic':[
  'kill yourself','die','go die','suicide','சாக','செத்துப் போ','சாவு','உன்னை சாக','saagu','seththu po','savu',
  'చచ్చిపో','చావు','నువ్వు చచ్చిపో','chachipo','chavu',
  'ಸಾಯು','ಸತ್ತು ಹೋಗು','saayu','sattu hogu',
  'ചാവൂ','ചത്തുപോ','chaavu','chathupo',
  'मर जा','मर जाओ','आत्महत्या','mar ja','mar jao','aatmahatya',
  'মরে যাও','মর','আত্মহত্যা','more jao','mor','atmohotya'
 ],
 'obscene':[
  'fuck','fucking','shit','bastard','asshole','what the fuck','damn',
  'புண்டை','தேவடியா','ஒழி','கேவலம்','போய் தொலை','போய்த்தொலை','மயிர்','மைரு','பொறுக்கி',
  'pundai','thevadiya','ozhi','kevalam','po tholai','poy tholai',
  'దరిద్రం','దెంగు','దెంగ','దెంగేయ్','చెత్త','వెధవ','daridram','dengu','denga','chetta',
  'ಕೊಳಕು','ಕೆಟ್ಟ ಪದ','ಹೋಗು','ನಾಯಿ','kolaku','kettu pada','hogu',
  'പന്നി','പന്നിയാണ്','തെറി','അശ്ലീലം','മൈര്','മൈരേ','myre','panni','theri','ashleelam',
  'भाड़ में जाओ','गाली','बकवास','भोसड़ी','चूत','गंदी गाली','fuck you','bhaad mein jao','gaali','bakwaas','bhosdi','choot',
  'চোদ','চোদা','গালি','নোংরা','বাজে কথা','জঘন্য গালি','chod','gali','nongra','baje kotha'
 ],
 'threat':[
  'i will find you','i will hurt you',"i'll hurt you",'hurt you','kill you','i will kill',"i'll kill",'beat you','attack you','you will regret',"you'll regret",'find you and hurt',
  'உன்னை கண்டுபிடித்து','உன்னை காயப்படுத்துவேன்','கொன்று விடுவேன்','உன்னை கொல்வேன்','தாக்குவேன்','அடிப்பேன்','காயப்படுத்துவேன்',
  'unnai kandupidiththu','unnai kaayapaduththuven','unnai kolven','kondru viduven','thaakkuven','adippen','kaayapaduththuven',
  'నిన్ను కనుక్కొని','నిన్ను చంపుతాను','నిన్ను కొడతాను','నిన్ను గాయపరుస్తాను','దాడి చేస్తాను',
  'ninnu kanukkoni','ninnu champuthanu','ninnu kodathanu','ninnu gayaparusthanu','dadi chestanu',
  'ನಿನ್ನನ್ನು ಹುಡುಕಿ','ನಿನ್ನನ್ನು ಕೊಲ್ಲುತ್ತೇನೆ','ನಿನ್ನನ್ನು ಹೊಡೆಯುತ್ತೇನೆ','ನಿನ್ನನ್ನು ಗಾಯಗೊಳಿಸುತ್ತೇನೆ','ದಾಳಿ ಮಾಡುತ್ತೇನೆ',
  'ninnannu huduki','ninnannu kolluttene','ninnannu hodeyuttene','ninnannu gayagolisuttene','daali maduttene',
  'നിന്നെ കണ്ടെത്തി','നിന്നെ കൊല്ലും','നിന്നെ അടിക്കും','നിന്നെ ഉപദ്രവിക്കും',
  'ninne kandethi','ninne kollum','ninne adikkum','ninne upadravikkum',
  'मैं तुम्हें ढूंढूंगा','मैं तुम्हें मार दूंगा','तुम्हें चोट पहुंचाऊंगा','तुम्हें पीटूंगा','तुम पर हमला करूंगा',
  'main tumhe dhundhunga','main tumhe maar dunga','tumhe chot pahuchaunga','tumhe peetunga','tum par hamla karunga',
  'আমি তোমাকে খুঁজে বের করব','আমি তোমাকে মেরে ফেলব','আমি তোমাকে আঘাত করব','তোমাকে মারব',
  'ami tomake khuje ber korbo','ami tomake mere felbo','ami tomake aghat korbo','tomake marbo'
 ],
 'insult':[
  'idiot','stupid','moron','loser','fool','dumb','jerk','clown','pathetic','asshole',
  'முட்டாள்','மூடன்','அறிவில்லாத','கேவலமானவன்','அசிங்கமானவன்','முட்டாள்தனமாக','முட்டாளே',
  'muttal','moodan','arivillatha','kevalamanavan','asingamanavan','muttale',
  'వెధవ','మూర్ఖుడు','పిచ్చివాడు','దద్దమ్మ','vedhava','murkhudu','pichivadu','daddamma',
  'ದಡ್ಡ','ಮೂರ್ಖ','ಹುಚ್ಚ','ಬೇವಕೂಫ್','dadda','moorkha','huchcha','bevakuf',
  'മണ്ടൻ','വിഡ്ഢി','ഭ്രാന്തൻ','പൊട്ടൻ','mandan','viddhi','bhranthan','pottan',
  'बेवकूफ','मूर्ख','गधा','पागल','नालायक','बदतमीज','bewakoof','moorkh','gadha','pagal','nalayak','badtameez',
  'বোকা','মূর্খ','গাধা','পাগল','অসভ্য','boka','murkho','gadha','pagol','oshobbo'
 ],
 'identity_hate':[
  'racist','race','religion','caste','ethnic','community','ஜாதி','மதம்','இனம்','சாதி','जाति','धर्म','नस्ल','समुदाय',
  'ജാതി','മതം','വംശം','ജാതിവെറി','ಜಾತಿ','ಧರ್ಮ','ವಂಶ','జాతి','మతం','కులం','জাতি','ধর্ম','বর্ণ'
 ]
}
POS=['great work','good work','well done','excellent','nice','thank you','thanks','awesome','good job','keep it up','அருமை','நல்ல வேலை','மிகவும் நன்று','நன்றி','சிறப்பு','வாழ்த்துகள்','చాలా బాగుంది','ధన్యవాదాలు','ಅದ್ಭುತ','ಧನ್ಯವಾದಗಳು','വളരെ നന്നായി','നന്ദി','बहुत अच्छा','धन्यवाद','খুব ভালো','ধন্যবাদ']

class MessageIn(BaseModel):
 text: Optional[str]=None; message: Optional[str]=None; context: str=''; context_messages:list[str]=Field(default_factory=list); user_id:str='anonymous'; conversation_id:str='default'
 def get_text(self): return (self.text if self.text is not None else self.message or '').strip()
 def get_context(self): return [str(x).strip() for x in self.context_messages if str(x).strip()] or [x.strip() for x in self.context.splitlines() if x.strip()]
class LoginIn(BaseModel): username:str; password:str; role:Optional[str]=None
class RegisterIn(BaseModel): username:str; password:str
class HumanDecisionIn(BaseModel): decision:str; note:str=''
class BlockIn(BaseModel): username:str

def now(): return datetime.now(timezone.utc).isoformat()
def db():
 c=sqlite3.connect(DB_PATH); c.row_factory=sqlite3.Row; return c
def hash_password(password,salt=None):
 salt=salt or secrets.token_hex(16); return hashlib.pbkdf2_hmac('sha256',password.encode(),salt.encode(),150000).hex(),salt
def verify_password(password,stored,salt): return secrets.compare_digest(hash_password(password,salt)[0],stored)
def init_db():
 c=db(); c.executescript('''CREATE TABLE IF NOT EXISTS users(id INTEGER PRIMARY KEY AUTOINCREMENT,username TEXT UNIQUE NOT NULL,password_hash TEXT NOT NULL,salt TEXT NOT NULL,role TEXT NOT NULL DEFAULT 'user',blocked INTEGER NOT NULL DEFAULT 0,created_at TEXT NOT NULL);CREATE TABLE IF NOT EXISTS moderation_results(id INTEGER PRIMARY KEY AUTOINCREMENT,user_id TEXT NOT NULL,conversation_id TEXT NOT NULL,message TEXT NOT NULL,result_json TEXT NOT NULL,created_at TEXT NOT NULL);CREATE TABLE IF NOT EXISTS audit_logs(id INTEGER PRIMARY KEY AUTOINCREMENT,user_id TEXT,action TEXT NOT NULL,details TEXT,created_at TEXT NOT NULL);''')
 admin_row=c.execute('SELECT id FROM users WHERE lower(username)=lower(?)',('admin',)).fetchone()
 if not admin_row:
  h,s=hash_password('Admin@1234'); c.execute('INSERT INTO users(username,password_hash,salt,role,created_at) VALUES(?,?,?,?,?)',('admin',h,s,'admin',now()))
 else:
  # The project uses one stable demo admin credential. This also repairs databases
  # created by earlier AURA builds that seeded a different password.
  h,s=hash_password('Admin@1234'); c.execute("UPDATE users SET password_hash=?, salt=?, role='admin', blocked=0 WHERE lower(username)=lower(?)",(h,s,'admin'))
 c.commit(); c.close()

def token_for(username,role):
 t=hashlib.sha256(f'{username}:{role}:{secrets.token_hex(16)}'.encode()).hexdigest(); SESSIONS[t]={'username':username,'role':role}; return t
def current(auth):
 if not auth or not auth.startswith('Bearer '): raise HTTPException(401,'Authentication required.')
 u=SESSIONS.get(auth.split(' ',1)[1].strip())
 if not u: raise HTTPException(401,'Session expired. Please sign in again.')
 return u
def admin(auth):
 u=current(auth)
 if u['role']!='admin': raise HTTPException(403,'Admin access required.')
 return u
def lang(text):
    scripts={
        'Tamil': r'[\u0B80-\u0BFF]', 'Telugu': r'[\u0C00-\u0C7F]',
        'Kannada': r'[\u0C80-\u0CFF]', 'Malayalam': r'[\u0D00-\u0D7F]',
        'Hindi': r'[\u0900-\u097F]', 'Bengali': r'[\u0980-\u09FF]'
    }
    counts={k:len(re.findall(v,text)) for k,v in scripts.items()}
    found=[k for k,v in counts.items() if v]
    latin=len(re.findall(r'[A-Za-z]',text))
    if len(found)==1:
        return found[0] + (' + English' if latin else '')
    if len(found)>1:
        return 'Mixed Indian Languages' + (' + English' if latin else '')
    low=norm(text)
    roman_markers={
      'Tamil':['muttal','moodan','kevalam','kevalamanavan','asingam','paithiyam','savu','seththu','unnai','po tholai'],
      'Telugu':['pichivadu','murkhudu','vedhava','chachipo','ninnu','dengu'],
      'Kannada':['huchcha','moorkha','dadda','saayu','ninnannu','hogu'],
      'Malayalam':['mandan','viddhi','bhranthan','chaavu','ninne','upadravikkum'],
      'Hindi':['bewakoof','moorkh','gadha','pagal','nalayak','mar ja','tumhe','dhundhunga'],
      'Bengali':['boka','murkho','gadha','pagol','more jao','tomake','khujе']
    }
    scores={k:sum(1 for m in ms if m in low) for k,ms in roman_markers.items()}
    best=max(scores,key=scores.get) if scores else 'English'
    return best if scores.get(best,0)>0 else 'English'

def norm(s):
    # Normalize whitespace and common punctuation; preserve native Unicode.
    s=s.lower().replace('’',"'").replace('“','"').replace('”','"')
    return re.sub(r'\s+',' ',s).strip()

def hits(text,terms):
    low=norm(text)
    # Word/phrase matching for Latin text plus direct Unicode phrase matching.
    out=[]
    for x in terms:
        term=norm(x)
        if not term: continue
        if re.search(r'[a-z0-9]',term):
            if re.search(r'(?<![a-z0-9])'+re.escape(term)+r'(?![a-z0-9])',low): out.append(x)
        elif term in low:
            out.append(x)
    return out

# WARN calibration: only mild insults/toxicity are lowered into the WARN band.
# Strong profanity, threats, identity hate and severe toxicity keep their existing rules.
MILD_TERMS = {
    'stupid','idiot','dumb','fool','jerk','clown','silly','annoying','rude','disrespectful',
    'முட்டாள்','மூடன்','அறிவில்லாத'
}
MILD_TOXIC_SCORE = 42.0
MILD_INSULT_SCORE = 45.0

# RESTRICT calibration: explicit repeated/targeted harassment is restricted even
# when individual insult words are mild. This is isolated from other decisions.
RESTRICT_PATTERNS = (
    # English
    r'\brepeatedly\b.*\b(target|targeting|targeted|harass|harassing|harassed|insult|insulting|attack|attacking)\b',
    r'\b(keep|keeps|kept|continue|continues|continued)\b.*\b(target|targeting|harass|harassing|insult|insulting|attack|attacking)\b',
    r'\b(every|multiple|several)\b.*\b(conversation|discussion|discussions|post|posts|message|messages)\b.*\b(target|targeting|harass|harassing|insult|insulting)\b',
    r'\basked\b.*\bstop\b.*\b(continue|continues|continued|keep|keeps|kept)\b.*\b(harass|harassing|target|targeting|insult|insulting)\b',
    r'\b(stop|leave me alone)\b.*\b(harass|harassing|target|targeting)\b.*\b(again|again and again|repeatedly)\b',
    # Tamil / Tanglish
    r'(தொடர்ந்து|மீண்டும் மீண்டும்|ஒவ்வொரு).*(குறிவைத்து|துன்புறுத்தி|அவமதித்து|தாக்கி)',
    r'(நிறுத்தச் சொன்ன|நிறுத்த சொன்ன|விட்டுவிடச் சொன்ன).*(தொடர்ந்து|மீண்டும்).*(துன்புறுத்த|அவமதிக்க|குறிவைக்க)',
    r'\b(continue|keep|repeatedly|again and again)\b.*\b(harass|target|insult)\b.*\b(ennai|unnai)\b',
    # Telugu / Tenglish
    r'(ప్రతిసారి|మళ్లీ మళ్లీ|తరచుగా).*(లక్ష్యంగా|వేధిస్తూ|అవమానిస్తూ|దాడి)',
    r'\b(pratisari|malli malli|tarachuga)\b.*\b(target|vedhistu|avamanistu|harass)\b',
    # Kannada / Kanglish
    r'(ಪ್ರತಿ ಬಾರಿ|ಮತ್ತೆ ಮತ್ತೆ|ಪದೇ ಪದೇ).*(ಗುರಿಯಾಗಿಸಿ|ಕಿರುಕುಳ|ಅವಮಾನಿಸಿ|ದಾಳಿ)',
    r'\b(prati bari|matte matte|pade pade)\b.*\b(target|kirukula|avamana|harass)\b',
    # Malayalam / Manglish
    r'(എപ്പോഴും|വീണ്ടും വീണ്ടും|തുടർച്ചയായി).*(ലക്ഷ്യമാക്കി|ഉപദ്രവിച്ച്|അപമാനിച്ച്|ആക്രമിച്ച്)',
    r'\b(eppozhum|veendum veendum|thudarccayayi)\b.*\b(target|upadrav|apaman|harass)\b',
    # Hindi / Hinglish
    r'(बार बार|लगातार|हर बातचीत).*(निशाना|परेशान|अपमान|हमला)',
    r'\b(bar bar|lagatar|har baat)\b.*\b(target|pareshan|apman|harass)\b',
    # Bengali / Banglish
    r'(বার বার|বারবার|প্রতিবার|ক্রমাগত).*(লক্ষ্য|হয়রানি|অপমান|আক্রমণ)',
    r'\b(bar bar|barbar|protibar|kromagoto)\b.*\b(target|hoyrani|opoman|harass)\b',
)

# ---------------------------------------------------------------------------
# Multilingual decision calibration
# ---------------------------------------------------------------------------
# The English rules above are intentionally left unchanged.  These rules add
# high-precision phrase coverage for Indian-script languages and their common
# Romanized/mixed forms.  They are decision cues, not a claim of model training.
MULTI_DECISION_PATTERNS = {
    'WARN': [
        # Tamil
        r'(நீ|நீங்கள்).*(முட்டாள்|மூடன்|முட்டாளே)',
        r'\bnee\b.*\b(muttal|moodan|muttale)\b',
        # Telugu
        r'(నువ్వు|నీవు).*(మూర్ఖుడు|మూర్ఖుడివి|వెధవ|వెధవా|వెధవుడివి|దద్దమ్మ)',
        r'\b(nuvvu|neevu)\b.*\b(murkhudu|murkhudivi|vedhava|vedhavodivi|daddamma)\b',
        # Kannada
        r'(ನೀನು|ನೀವು).*(ಮೂರ್ಖ|ದಡ್ಡ|ಬೇವಕೂಫ್)',
        r'\b(neenu|neevu)\b.*\b(moorkha|dadda|bevakuf)\b',
        # Malayalam
        r'(നീ|നിങ്ങൾ).*(മണ്ടൻ|മണ്ടനാണ്|വിഡ്ഢി|വിഡ്ഢിയാണ്|പൊട്ടൻ|പൊട്ടനാണ്)',
        r'\b(nee|ningal)\b.*\b(mandan|mandananu|viddhi|viddhiyanu|pottan|pottananu)\b',
        # Hindi
        r'(तुम|आप).*(बेवकूफ|मूर्ख|गधा|नालायक)',
        r'\b(tum|aap)\b.*\b(bewakoof|moorkh|gadha|nalayak)\b',
        # Bengali
        r'(তুমি|আপনি).*(বোকা|মূর্খ|গাধা)',
        r'\b(tumi|apni)\b.*\b(boka|murkho|gadha)\b',
    ],
    'QUARANTINE': [
        # Tamil
        r'(கேவலமானவன்|கேவலமானவள்|அசிங்கமானவன்|அசிங்கமானவள்|தேவடியா|புண்டை|பொறுக்கி|போய் தொலை)',
        r'\b(romba|miga|migavum)?\s*(kevalama|asingama)\s*(pesura|irukka|irukkiya)\b',
        r'\b(kevalamanavan|asingamanavan|thevadiya|pundai|porukki)\b',
        # Telugu
        r'(దరిద్రం|దెంగు|దెంగేయ్|చెత్త|నీచుడు|దరిద్రుడు|అసహ్యంగా)',
        r'\b(chala|chaala)\b.*\b(disgusting|daridram|dengu|chetta|asahyam|asahyanga)\b',
        r'\b(daridram|dengu|denga|chetta)\b',
        # Kannada
        r'(ಕೊಳಕು|ಅಸಹ್ಯ|ನೀಚ|ಕೆಟ್ಟವನು)',
        r'\b(tumba|thumba)\b.*\b(asahya|kolaku|neech)\b',
        r'\b(asahya|asahya|asahyakar|kolaku|neech)\b',
        # Malayalam
        r'(വൃത്തികെട്ടവൻ|വൃത്തികെട്ടവനാണ്|വൃത്തികെട്ടവൾ|തെറി|മൈര്|മൈരേ|അശ്ലീലം)',
        r'\b(valare)\b.*\b(disgusting|vrithikettavan|vrithikettavananu|vrithikettavanan|myre)\b',
        r'\b(vrithikettavan|myre|theri)\b',
        # Hindi
        r'(भोसड़ी|चूत|भाड़ में जाओ|गंदी गाली|घटिया|जघन्य)',
        r'\b(bahut)\b.*\b(disgusting|ghatiya|bhosdi|choot)\b',
        r'\b(bhosdi|choot|ghatiya|ghatiya|bakwaas)\b',
        # Bengali
        r'(চোদা|চোদ|নোংরা|জঘন্য গালি|গালাগালি)',
        r'\b(choda|chod|nongra|joghonyo)\b',
        # Strong mixed profanity
        r'\b(fucking|fuck|shit)\b.*\b(kevalam|asingam|daridram|disgusting|ghatiya|myre)\b',
    ],
    'RESTRICT': [
        # Tamil / Tanglish
        r'(தொடர்ந்து|மீண்டும் மீண்டும்|பதிலுக்கு பதில்|ஒவ்வொரு முறையும்).*(துன்புறுத்த|அவமதிக்க|குறிவைக்க|தாக்க)',
        r'(என்னை|உன்னை).*(தொடர்ந்து|மீண்டும் மீண்டும்).*(துன்புறுத்த|அவமதிக்க|குறிவைக்க)',
        r'\b(nee|neenga)\b.*\b(ennai|enna|unnai|unna)\b.*\b(thodarnthu|meendum meendum|thodarchiyaga)\b.*\b(harass|target|insult|torture|panra|seyyura)\b',
        r'\b(thodarnthu|meendum meendum|thodarchiyaga)\b.*\b(ennai|enna|unnai|unna)\b.*\b(harass|target|insult|panra|seyyura)\b',
        # Telugu / Tenglish
        r'(ప్రతిసారి|మళ్లీ మళ్లీ|పదే పదే|తరచుగా).*(వేధ|అవమాన|లక్ష్య|దాడి)',
        r'(నన్ను|నిన్ను).*(మళ్లీ మళ్లీ|తరచుగా).*(వేధ|అవమాన|లక్ష్య)',
        r'\b(nuvvu|neevu)\b.*\b(nannu|ninnu)\b.*\b(malli malli|padhe padhe|tarachuga)\b.*\b(vedhi|vedhistu|harass|target|insult)\w*\b',
        r'\b(malli malli|padhe padhe|tarachuga)\b.*\b(nannu|ninnu)\b.*\b(harass|target|insult)\w*\b',
        # Kannada / Kanglish
        r'(ಪ್ರತಿ ಬಾರಿ|ಮತ್ತೆ ಮತ್ತೆ|ಪದೇ ಪದೇ|ನಿರಂತರವಾಗಿ).*(ಕಿರುಕುಳ|ಅವಮಾನ|ಗುರಿ|ದಾಳಿ)',
        r'(ನನ್ನನ್ನು|ನಿನ್ನನ್ನು).*(ಮತ್ತೆ ಮತ್ತೆ|ಪದೇ ಪದೇ).*(ಕಿರುಕುಳ|ಅವಮಾನ|ಗುರಿ)',
        r'\b(neenu|neevu)\b.*\b(nannannu|ninnannu)\b.*\b(matte matte|pade pade|nirantaravagi)\b.*\b(kirukula|harass|target|insult)\w*\b',
        # Malayalam / Manglish
        r'(വീണ്ടും വീണ്ടും|തുടർച്ചയായി|എപ്പോഴും).*(ഉപദ്രവ|അപമാന|ലക്ഷ്യമാക്കി|ആക്രമ)',
        r'(എന്നെ|നിന്നെ).*(വീണ്ടും വീണ്ടും|തുടർച്ചയായി).*(ഉപദ്രവ|അപമാന|ലക്ഷ്യ)',
        r'\b(nee|ningal)\b.*\b(enne|ninne)\b.*\b(veendum veendum|thudarccayayi|eppozhum)\b.*\b(upadrav|harass|target|insult)\w*\b',
        # Hindi / Hinglish
        r'(बार बार|बार-बार|लगातार|हर बार).*(परेशान|अपमान|निशाना|हमला)',
        r'(मुझे|तुम्हें).*(बार बार|लगातार).*(परेशान|अपमान|निशाना)',
        r'\b(tum|aap)\b.*\b(mujhe|tumhe)\b.*\b(bar bar|baar baar|lagatar|har baar)\b.*\b(pareshan|harass|target|insult)\w*\b',
        r'\b(tum|aap)\b.*\b(mujhe|mujhko|tumhe)\b.*\b(bar bar|baar baar|barbar|lagatar|har baar)\b.*\b(pareshan|harass|target|insult)\w*\b',
        r'\b(tumi|apni)\b.*\b(amake|tomake)\b.*\b(bar bar|barbar|protibar|kromagoto)\b.*\b(harass|target|insult)\w*\b',
        # Bengali / Banglish
        r'(বার বার|বারবার|প্রতিবার|ক্রমাগত).*(হয়রানি|অপমান|লক্ষ্য|আক্রমণ)',
        r'\b(tumi|apni)\b.*\b(amake|tomake)\b.*\b(bar bar|barbar|protibar|kromagoto)\b.*\b(harass|target|insult)\w*\b',
    ],
    'TEMPORARY_BLOCK': [
        # Native-script direct threats
        r'(நான்|நீ).*(காயப்படுத்துவேன்|கொன்றுவிடுவேன்|கொல்வேன்|தாக்குவேன்|அடிப்பேன்)',
        r'(నేను|నువ్వు).*(గాయపరుస్తాను|చంపుతాను|కొడతాను|దాడి చేస్తాను)',
        r'(ನಾನು|ನೀನು).*(ಗಾಯಗೊಳಿಸುತ್ತೇನೆ|ಕೊಲ್ಲುತ್ತೇನೆ|ಹೊಡೆಯುತ್ತೇನೆ|ದಾಳಿ ಮಾಡುತ್ತೇನೆ)',
        r'(ഞാൻ|നീ).*(പരിക്കേൽപ്പിക്കും|കൊല്ലും|അടിക്കും|ആക്രമിക്കും)',
        r'(मैं|तुम).*(मार दूंगा|मार दूँगा|चोट पहुँचाऊंगा|हमला करूंगा|पीटूंगा)',
        r'(আমি|তুমি).*(মেরে ফেলব|আঘাত করব|মারব|হামলা করব)',
        # Romanized
        r'\b(naan|nenu|nanu|njān|main|ami)\b.*\b(unnai|ninnu|ninnannu|ninne|tumhe|tomake)\b.*\b(kayapaduththuven|champuthanu|gayagolisuttene|kollum|maar dunga|hurt|kill|attack|marbo)\b',
        r'\b(i will|i am going to)\b.*\b(hurt|kill|attack|beat)\b',
    ],
    'HUMAN_REVIEW': [
        # Ambiguous, consequential but non-explicit statements
        r'(ஏதாவது|ஏதோ).*(நடக்கலாம்|நடக்கும்|நடக்குமோ)',
        r'(இப்படியே|தொடர்ந்தால்).*(ஏதாவது|ஏதோ).*(நடக்கலாம்|நடக்கும்)',
        r'\b(edhavadhu|edho)\b.*\b(nadakkalam|nadakkum|nadakkumo)\b',
        r'(ఏదైనా|ఏదో).*(జరగొచ్చు|జరగవచ్చు|జరుగుతుంది)',
        r'\b(edaina|edo)\b.*\b(jaragochu|jaragavachu|jarugutundi)\b',
        r'(ಏನಾದರೂ|ಏನೋ).*(ಆಗಬಹುದು|ನಡೆಯಬಹುದು|ನಡೆಯುತ್ತದೆ)',
        r'\b(enadaru|eno)\b.*\b(agabahudu|nadeyabahudu|nadeyutte)\b',
        r'(എന്തെങ്കിലും|എന്തോ).*(സംഭവിക്കാം|സംഭവിക്കും)',
        r'\b(enthenkilum|entho)\b.*\b(sambhavikkam|sambhavikkum)\b',
        r'(ശായദ്|शायद|कुछ).*(हो सकता है|होगा|हो जाए)',
        r'\b(shayad|kuch)\b.*\b(ho sakta hai|hoga|ho jaye)\b',
        r'(হয়তো|কিছু).*(ঘটতে পারে|হবে)',
        r'\b(hoyto|kichu)\b.*\b(ghotbe|hobe|ghot te pare)\b',
    ],
}

def multilingual_decision(text):
    """Return a high-confidence non-English decision cue, if present."""
    t=norm(text)
    # Do not alter the already-working English-only behaviour.
    native_scripts = bool(re.search(r'[\u0900-\u0DFF]', t))
    roman_markers = (
        'muttal','moodan','kevalam','kevalama','kevalamanavan','asingam','thodarnthu','meendum meendum',
        'ennai','unnai','harass','thodarchiyaga',
        'vedhava','pichivadu','nuvvu','nannu','ninnu','malli malli','tarachuga','asahya','disgusting',
        'moorkha','dadda','neenu','nannannu','ninnannu','matte matte','pade pade',
        'mandan','mandananu','viddhi','nee','enne','ninne','veendum veendum','thudarccayayi','valare',
        'bewakoof','moorkh','gadha','ghatiya','tum','mujhe','tumhe','bar bar','baar baar','lagatar',
        'boka','murkho','gadha','tumi','amake','tomake','barbar'
    )
    if not native_scripts and not any(m in t for m in roman_markers):
        return None
    for decision in ('TEMPORARY_BLOCK','RESTRICT','QUARANTINE','WARN','HUMAN_REVIEW'):
        if any(re.search(p,t) for p in MULTI_DECISION_PATTERNS[decision]):
            return decision
    return None

def classify(text):
    found={}; scores={x:0.0 for x in LABELS}
    base={'toxic':72,'severe_toxic':92,'obscene':82,'threat':98,'insult':78,'identity_hate':88}
    low=norm(text)
    for label in LABELS:
        h=hits(text,LEX[label])
        if not h:
            continue
        found[label]=h
        if label in {'toxic','insult'} and all(term.lower() in MILD_TERMS for term in h):
            scores[label]=min(100.0, (MILD_TOXIC_SCORE if label=='toxic' else MILD_INSULT_SCORE) + max(0,len(h)-1)*3)
        else:
            scores[label]=min(100.0,base[label]+max(0,len(h)-1)*4)
    return scores,found

def is_ambiguous_for_review(text, scores):
    # Human review is reserved for vague statements that imply a possible
    # consequence but do not contain an explicit harmful signal.
    if max(scores.values()) > 0:
        return False
    t=norm(text)
    ambiguous_patterns=(
        # English
        r'\bmaybe\b.*\b(happen|happens|something|consequence|result)\b',
        r'\b(might|could|may|probably)\b.*\b(happen|happens|something|consequence|result)\b',
        r'\b(something|something bad)\b.*\b(will|might|could|may)\b',
        r'\b(we\s+will\s+see|you\s+will\s+see)\s+what\s+happens\b',
        # Tamil / Tanglish
        r'(ஏதாவது|ஏதோ).*(நடக்கலாம்|நடக்கும்|நடக்குமோ)',
        r'(இப்படியே|தொடர்ந்தால்).*(ஏதாவது|ஏதோ).*(நடக்கலாம்|நடக்கும்)',
        r'(இப்படி தொடர்ந்தால்).*(ஏதாவது).*(நடக்கும்)',
        r'\b(edhavadhu|edho).*(nadakkalam|nadakkum)',
        r'\b(ipadiye|thodarnthal).*(edhavadhu|edho).*(nadakkalam|nadakkum)',
        # Telugu / Tenglish
        r'(ఏదైనా|ఏదో).*(జరగొచ్చు|జరుగుతుంది)',
        r'(ఇలాగే|కొనసాగితే).*(ఏదైనా|ఏదో).*(జరగొచ్చు)',
        r'\b(edaina|edo).*(jaragochu|jarugutundi)',
        # Kannada / Kanglish
        r'(ಏನಾದರೂ|ಏನೋ).*(ಆಗಬಹುದು|ನಡೆಯಬಹುದು|ನಡೆಯುತ್ತದೆ)',
        r'\b(enadaru|eno).*(agabahudu|nadeyabahudu|nadeyutte)',
        # Malayalam / Manglish
        r'(എന്തെങ്കിലും|എന്തോ).*(സംഭവിക്കാം|സംഭവിക്കും)',
        r'\b(enthenkilum|entho).*(sambhavikkam|sambhavikkum)',
        # Hindi / Hinglish
        r'(शायद|कुछ).*(हो सकता है|होगा|हो जाए)',
        r'(ऐसा ही चलता रहा|जारी रहा).*(कुछ|कुछ न कुछ).*(हो सकता है|होगा)',
        r'\b(shayad|kuch).*(ho sakta hai|hoga|ho jaye)',
        r'\b(tum|aap)\b.*\b(mujhe|mujhko|tumhe)\b.*\b(bar bar|baar baar|barbar|lagatar|har baar)\b.*\b(pareshan|harass|target|insult)\w*\b',
        r'\b(tumi|apni)\b.*\b(amake|tomake)\b.*\b(bar bar|barbar|protibar|kromagoto)\b.*\b(harass|target|insult)\w*\b',
        # Bengali / Banglish
        r'(হয়তো|কিছু).*(ঘটতে পারে|হবে)',
        r'\b(hoyto|kichu).*(ghot te pare|ghotbe|hobe)',
    )
    return any(re.search(p,t) for p in ambiguous_patterns)

def analyse(req):
 text=req.get_text(); ctx=req.get_context(); language=lang(text); scores,found=classify(text)
 multilingual_cue=multilingual_decision(text)
 if multilingual_cue:
  # Raise only the corresponding signal enough to land in its intended band.
  if multilingual_cue=='WARN':
   scores['toxic']=max(scores['toxic'],42.0); scores['insult']=max(scores['insult'],45.0)
  elif multilingual_cue=='QUARANTINE':
   scores['toxic']=max(scores['toxic'],72.0); scores['obscene']=max(scores['obscene'],82.0)
  elif multilingual_cue=='RESTRICT':
   scores['toxic']=max(scores['toxic'],72.0); scores['insult']=max(scores['insult'],78.0)
  elif multilingual_cue=='TEMPORARY_BLOCK':
   scores['threat']=max(scores['threat'],98.0)
  elif multilingual_cue=='HUMAN_REVIEW':
   scores={k:0.0 for k in LABELS}; found={}
 positive=hits(text,POS)
 if positive and not found: scores={x:0.0 for x in LABELS}
 repeated=sum(norm(x)==norm(text) for x in ctx)
 previous=sum(bool(hits(x,LEX['threat']) or hits(x,LEX['insult'])) for x in ctx)
 behaviour=min(100.0,previous*35+max(0,len(ctx)-5)*4)
 harassment=90.0 if scores['threat']>0 else (82.0 if scores['insult']>0 and previous else (60.0 if previous else 0.0))
 escalation=min(100.0,harassment+len(ctx)*2)
 safety=max(scores.values())
 confidence=96.0 if safety>=90 else (93.0 if safety>0 else (86.0 if len(text.split())<=2 else 91.0)); confidence-=2.0 if language=='Tamil + English' else 0
 context=min(100.0,repeated*35.0)
 score=safety*.72+context*.06+behaviour*.08+harassment*.10+escalation*.04
 if safety==0: score=min(14.0,context*.15+behaviour*.10)
 restrict_match=any(re.search(p, norm(text)) for p in RESTRICT_PATTERNS)
 if restrict_match: score=max(score,76.0)
 if multilingual_cue=='HUMAN_REVIEW': decision=('HUMAN_REVIEW','Human Review','Uncertain')
 elif multilingual_cue=='TEMPORARY_BLOCK': decision=('TEMPORARY_BLOCK','Temporary Block','Critical')
 elif multilingual_cue=='RESTRICT': decision=('RESTRICT','Restrict','High')
 elif multilingual_cue=='QUARANTINE': decision=('QUARANTINE','Quarantine','High')
 elif multilingual_cue=='WARN': decision=('WARN','Warn','Medium')
 elif is_ambiguous_for_review(text, scores): decision=('HUMAN_REVIEW','Human Review','Uncertain')
 elif scores['threat']>=90 or scores['severe_toxic']>=90: decision=('TEMPORARY_BLOCK','Temporary Block','Critical')
 elif restrict_match or scores['identity_hate']>=85 or score>=75: decision=('RESTRICT','Restrict','High')
 elif score>=55: decision=('QUARANTINE','Quarantine','High')
 elif score>=30: decision=('WARN','Warn','Medium')
 else: decision=('ALLOW','Allow','Safe' if score==0 else 'Low')
 action,label,rlevel=decision
 evidence=[f'{k.replace("_"," ").title()}: {v:.1f}%' for k,v in sorted(scores.items(),key=lambda x:x[1],reverse=True) if v>0][:3] or ['No harmful safety signal detected.']
 action_public = {'ALLOW':'ALLOW','WARN':'WARN','QUARANTINE':'QUARANTINE','RESTRICT':'RESTRICT','TEMPORARY_BLOCK':'TEMPORARY_BLOCK','HUMAN_REVIEW':'HUMAN_REVIEW'}[action]
 probabilities={k:round(v,1) for k,v in scores.items()}
 confidence_data={'confidence':round(max(0,confidence),1),'uncertainty':round(100-max(0,confidence),1)}
 return {'language':language,'classification':scores,'classification_matches':found,'context_analysis':{'message_count':len(ctx),'repeated_current_message':repeated>0,'context_risk':round(context,1)},'behavioural_analysis':{'previous_violations':previous,'recent_frequency':len(ctx),'behavioural_risk':round(behaviour,1)},'harassment_escalation':{'harassment_risk':round(harassment,1),'escalation_risk':round(escalation,1),'repeated_targeting':bool(previous)},'behaviour':{'risk':round(behaviour,1),'previous_violations':previous,'recent_frequency':len(ctx)},'harassment_analysis':{'harassment_risk':round(harassment,1),'escalation_risk':round(escalation,1),'repeated_targeting':bool(previous)},'escalation_analysis':{'risk':round(escalation,1)},'confidence_uncertainty':{'confidence':round(max(0,confidence),1),'uncertainty':round(100-max(0,confidence),1),'uncertain':False},'risk':{'score':round(min(100,score),1),'overall':round(min(100,score),1),'context':round(context,1),'behavioural':round(behaviour,1),'harassment':round(harassment,1),'escalation':round(escalation,1),'confidence':round(max(0,confidence),1)},'risk_score':round(min(100,score),1),'risk_level':rlevel,'decision':{'action':action,'label':label,'risk_level':rlevel},'moderation':{'action':action,'label':label,'risk_level':rlevel},'action':action,'probabilities':probabilities,'confidence':round(max(0,confidence),1),'uncertainty':round(100-max(0,confidence),1),'xai':{'important_evidence':'; '.join(evidence),'summary':f'AURA selected {label} from safety signals, context, behaviour and confidence.','reason':'No harmful signal was detected, so the comment can be posted.' if action=='ALLOW' else ('A direct threat or severe harmful signal was detected, so the account action is strengthened.' if action=='TEMPORARY_BLOCK' else f'The combined risk score is {score:.1f}/100, leading to {label}.')},'explainable_ai':{'important_evidence':'; '.join(evidence),'summary':f'AURA selected {label} from safety signals, context, behaviour and confidence.','reason':'No harmful signal was detected, so the comment can be posted.' if action=='ALLOW' else ('A direct threat or severe harmful signal was detected, so the account action is strengthened.' if action=='TEMPORARY_BLOCK' else f'The combined risk score is {score:.1f}/100, leading to {label}.')}}

def result(req):
 if not req.get_text(): raise HTTPException(400,'Please enter a comment before running the safety check.')
 r=analyse(req); action=r['decision']['action']; r.update({'message_id':hashlib.sha256(f'{req.user_id}|{req.conversation_id}|{req.get_text()}'.encode()).hexdigest()[:16],'user_id':req.user_id,'conversation_id':req.conversation_id,'original_message':req.get_text(),'preprocessed_message':req.get_text(),'publication':{'status':'POSTED' if action=='ALLOW' else ('HUMAN_REVIEW' if action=='HUMAN_REVIEW' else 'NOT_POSTED'),'posted':action=='ALLOW','flagged':action not in {'ALLOW','HUMAN_REVIEW'},'needs_human':action=='HUMAN_REVIEW'},'created_at':now()}); return r

@app.on_event('startup')
def startup(): init_db()
@app.get('/')
def root(): return {'name':'AURA','status':'online','version':app.version}
@app.get('/health')
def health(): return {'status':'healthy','service':'aura-backend'}
@app.post('/auth/login')
def login(req:LoginIn):
 c=db(); row=c.execute('SELECT * FROM users WHERE lower(username)=lower(?)',(req.username.strip(),)).fetchone(); c.close()
 expected_role=req.role or ('admin' if req.username.strip().lower()=='admin' else 'user')
 if not row or row['blocked'] or row['role']!=expected_role or not verify_password(req.password,row['password_hash'],row['salt']): raise HTTPException(401,'Invalid credentials.')
 tok=token_for(row['username'],row['role']); return {'token':tok,'access_token':tok,'user':{'id':row['id'],'username':row['username'],'role':row['role']}}
@app.post('/auth/register')
def register(req:RegisterIn):
 u=req.username.strip()
 if len(u)<3: raise HTTPException(400,'Username must contain at least 3 characters.')
 if len(req.password)<6: raise HTTPException(400,'Password must contain at least 6 characters.')
 h,s=hash_password(req.password); c=db()
 try: c.execute('INSERT INTO users(username,password_hash,salt,role,created_at) VALUES(?,?,?,?,?)',(u,h,s,'user',now())); c.commit()
 except sqlite3.IntegrityError: c.close(); raise HTTPException(409,'Username already exists.')
 c.close(); tok=token_for(u,'user'); return {'message':'Account created successfully.','access_token':tok,'token':tok,'user':{'username':u,'role':'user'}}
@app.post('/moderate')
def moderate(req:MessageIn,authorization:Optional[str]=Header(default=None)):
 u=current(authorization); req.user_id=u['username']; r=result(req); c=db(); cur=c.execute('INSERT INTO moderation_results(user_id,conversation_id,message,result_json,created_at) VALUES(?,?,?,?,?)',(u['username'],req.conversation_id,req.get_text(),json.dumps(r,ensure_ascii=False),r['created_at'])); mid=cur.lastrowid; c.execute('INSERT INTO audit_logs(user_id,action,details,created_at) VALUES(?,?,?,?)',(u['username'],'MODERATION_DECISION',json.dumps({'moderation_id':mid,'decision':r['decision']}),now())); c.commit(); c.close(); r['moderation_id']=mid; return r
@app.get('/history')
def history(authorization:Optional[str]=Header(default=None)):
 u=current(authorization); c=db(); rows=c.execute('SELECT id,conversation_id,message,result_json,created_at FROM moderation_results WHERE user_id=? ORDER BY id DESC LIMIT 50',(u['username'],)).fetchall(); c.close(); return [{'id':r['id'],'conversation_id':r['conversation_id'],'message':r['message'],'result':json.loads(r['result_json']),'created_at':r['created_at']} for r in rows]
@app.get('/comments/me')
def comments_me(authorization:Optional[str]=Header(default=None)):
 u=current(authorization); c=db(); rows=c.execute('SELECT id,message,result_json,created_at FROM moderation_results WHERE user_id=? ORDER BY id DESC LIMIT 50',(u['username'],)).fetchall(); c.close()
 out=[]
 for r in rows:
  x=json.loads(r['result_json']); out.append({'id':r['id'],'text':r['message'],'action':x['decision']['action'],'status':x['publication']['status'],'risk_score':x['risk_score'],'created_at':r['created_at']})
 return out

@app.get('/admin/overview')
def admin_overview(authorization:Optional[str]=Header(default=None)):
 admin(authorization)
 c=db(); users_rows=c.execute('SELECT id,username,role,blocked,created_at FROM users ORDER BY id').fetchall(); event_rows=c.execute('SELECT id,user_id,message,result_json,created_at FROM moderation_results ORDER BY id DESC LIMIT 100').fetchall(); c.close()
 events=[{'moderation_id':r['id'],'user_id':r['user_id'],'message':r['message'],'result':json.loads(r['result_json']),'created_at':r['created_at']} for r in event_rows]
 pending=sum(1 for e in events if e['result'].get('publication',{}).get('needs_human'))
 return {'users':[dict(r) for r in users_rows],'events':events,'pending_human_reviews':pending}

def admin_rows():
 c=db(); rows=c.execute('SELECT id,user_id,message,result_json,created_at FROM moderation_results ORDER BY id DESC LIMIT 100').fetchall(); c.close(); return [{'moderation_id':r['id'],'user_id':r['user_id'],'message':r['message'],'result':json.loads(r['result_json']),'created_at':r['created_at']} for r in rows]
@app.get('/admin/comments')
def comments(authorization:Optional[str]=Header(default=None)): admin(authorization); return admin_rows()
@app.get('/admin/human-review')
def review(authorization:Optional[str]=Header(default=None)): admin(authorization); return [x for x in admin_rows() if x['result']['publication']['needs_human']]
@app.post('/admin/human-review/{mid}')
def human(mid:int,req:HumanDecisionIn,authorization:Optional[str]=Header(default=None)):
 a=admin(authorization); d=req.decision.upper(); c=db(); row=c.execute('SELECT result_json FROM moderation_results WHERE id=?',(mid,)).fetchone()
 if not row: c.close(); raise HTTPException(404,'Moderation item not found.')
 r=json.loads(row['result_json'])
 if d=='APPROVE': r['decision']={'action':'ALLOW','label':'Allow','risk_level':r['risk_level']}; r['publication']={'status':'POSTED','posted':True,'flagged':False,'needs_human':False}
 elif d=='RESTRICT': r['decision']={'action':'RESTRICT','label':'Restrict','risk_level':'High'}; r['publication']={'status':'NOT_POSTED','posted':False,'flagged':True,'needs_human':False}
 elif d in {'REJECT','BLOCK','DISMISS'}: r['decision']={'action':'TEMPORARY_BLOCK' if d=='BLOCK' else 'QUARANTINE','label':'Temporary Block' if d=='BLOCK' else 'Quarantine','risk_level':'Critical' if d=='BLOCK' else r['risk_level']}; r['publication']={'status':'NOT_POSTED','posted':False,'flagged':True,'needs_human':False}
 else: c.close(); raise HTTPException(400,'Unsupported human-review decision.')
 cur=c.execute('UPDATE moderation_results SET result_json=? WHERE id=?',(json.dumps(r,ensure_ascii=False),mid))
 if cur.rowcount != 1:
  c.rollback(); c.close(); raise HTTPException(500,'Unable to save the human-review decision.')
 c.execute('INSERT INTO audit_logs(user_id,action,details,created_at) VALUES(?,?,?,?)',(a['username'],'HUMAN_REVIEW',json.dumps({'moderation_id':mid,'decision':d}),now()))
 c.commit(); c.close(); return {'message':'Human decision saved.','moderation_id':mid,'result':r}
@app.get('/admin/users')
def users(authorization:Optional[str]=Header(default=None)):
 admin(authorization); c=db(); rows=c.execute('SELECT id,username,role,blocked,created_at FROM users ORDER BY id').fetchall(); c.close(); return [dict(r) for r in rows]
@app.post('/admin/block-user')
def block(req:BlockIn,authorization:Optional[str]=Header(default=None)):
 a=admin(authorization); u=req.username.strip()
 if u.lower()=='admin': raise HTTPException(400,'The admin account cannot be blocked.')
 c=db(); cur=c.execute('UPDATE users SET blocked=1 WHERE lower(username)=lower(?)',(u,));
 if cur.rowcount==0: c.close(); raise HTTPException(404,'User not found.')
 c.execute('INSERT INTO audit_logs(user_id,action,details,created_at) VALUES(?,?,?,?)',(a['username'],'BLOCK_USER',json.dumps({'username':u}),now())); c.commit(); c.close(); return {'message':f'User {u} blocked.'}


NameError: name '__file__' is not defined